# Crop recommendation: XGBoost training, scores and the map dashboard

Team 8, Part B (Thierry). Agreed at the stand-up of 28 Oct 2026: train XGBoost, give its scores next to the other models, and show it on a map where you click a field and get the crop for that point.

**What this notebook does** (about 5 minutes on a normal CPU runtime, no GPU needed):

1. Gets the code and the data from GitHub.
2. Installs the library versions of the official evaluation, in their own environment.
3. Looks at the training data.
4. Runs the official evaluation (every model of the protocol) and checks it against the committed results.
5. Trains XGBoost with the protocol's two tests and shows its scores.
6. Trains the final XGBoost on all 1,904 fields and shows what it relies on.
7. Tries a few points the way the dashboard does.
8. Saves the results.
9. Runs all the tests.
10. Opens the dashboard with a link.

**To run it:** open the Runtime menu and choose Run all. If Colab warns that the notebook was not written by Google, choose Run anyway.

**For teammates (Gabin and the others):** run it in your own Colab. It needs nothing from Thierry's laptop or Drive. Before you change a cell, make your own copy (File menu, Save a copy in Drive) so the shared notebook stays clean.

**Privacy:** the clone takes only the latest commit, without the old history, and the dashboard shows 2 km blocks, never a farm's GPS point. Never paste a token or a password into a cell: use Colab secrets (the key icon on the left).

In [ ]:
# Settings. Change BRANCH only to run code that is not merged into main yet, for example "feat/map-xgboost".
REPO_URL = "https://github.com/ishimwethierryhenry/agri-decision-support-rwanda.git"
BRANCH = "main"
REPO = "/content/agri-decision-support-rwanda"
MODULE = f"{REPO}/01-crop-recommendation"
VENV = "/content/venv"
PY = f"{VENV}/bin/python"

## 1. Get the repo

Tried in this order:

1. A public clone (the repo is public at the moment).
2. If the repo is private again: a clone with a GitHub token saved in Colab secrets under the name `GITHUB_TOKEN` (key icon on the left, add the secret, turn on Notebook access). The token is never printed and is removed from the clone right after.
3. If neither works: upload a zip of the repo made on your laptop, in the repo folder, with `git archive --format=zip -o agri-repo.zip HEAD`.

In [ ]:
import os, shutil, subprocess

def clone(url):
    shutil.rmtree(REPO, ignore_errors=True)
    r = subprocess.run(["git", "clone", "-q", "--depth", "1", "--branch", BRANCH, url, REPO],
                       capture_output=True, text=True)
    return r.returncode == 0

how = None
if clone(REPO_URL):
    how = "public clone"
else:
    print(f"No public clone of branch {BRANCH} (the repo may be private, or the branch name may be wrong).")
    try:
        from google.colab import userdata
        token = userdata.get("GITHUB_TOKEN")
    except Exception:
        token = None
    if token and clone(REPO_URL.replace("https://", f"https://x-access-token:{token}@")):
        subprocess.run(["git", "-C", REPO, "remote", "set-url", "origin", REPO_URL], check=True)
        how = "clone with the GITHUB_TOKEN secret"
    token = None
if how is None:
    print("No GITHUB_TOKEN secret either. Upload agri-repo.zip (made with git archive, see above).")
    from google.colab import files
    name = next(iter(files.upload()))
    shutil.rmtree(REPO, ignore_errors=True)
    subprocess.run(["unzip", "-q", "-o", name, "-d", REPO], check=True)
    how = "uploaded zip"
last = subprocess.run(["git", "-C", REPO, "log", "-1", "--format=%h %s (%cd)", "--date=short"],
                      capture_output=True, text=True).stdout.strip()
print(f"Repo ready ({how}). Code: {last or 'from a zip, no git history'}")

## 2. The same library versions as the official evaluation

Other versions move the numbers: scikit-learn changes the 2 km folds and XGBoost changes its trees. So the models run in their own environment with the pinned versions (`01-crop-recommendation/requirements-eval.txt`, plus the dashboard libraries). The notebook's own Python only shows the tables and the charts. About one minute.

In [ ]:
!pip install -q --root-user-action=ignore uv
!rm -rf {VENV} && uv venv -q -p 3.12 {VENV}
!uv pip install -q -p {PY} -r {REPO}/03-shared/dashboard/requirements.txt pytest
!{PY} -c "import sys, sklearn, xgboost, pandas, numpy, streamlit; print('python', sys.version.split()[0], '| scikit-learn', sklearn.__version__, '| xgboost', xgboost.__version__, '| pandas', pandas.__version__, '| numpy', numpy.__version__, '| streamlit', streamlit.__version__)"

## 3. The training data

The table Part A built (`feature_table_v2.csv`): one row per CIAT field of season B 2025, in Musanze, Nyabihu, Nyagatare and Ruhango. A field is a training field when its GPS passed Part A's checks (`qc_ok`) and it has one of the four single crops.

Inputs the models may use (the code refuses any other column):

- elevation of the field;
- season weather for the district (NASA POWER): rainfall, maximum, minimum and mean temperature, sunlight;
- the district temperature moved to the field's height (6.5 C per 1,000 m);
- NISR crop shares of the district;
- soil at 0 to 20 cm (iSDAsoil): nitrogen, phosphorus, potassium, pH, sand, silt, clay, organic carbon, CEC.

Never used: anything about the crop already in the field, IDs, the district name, and CIAT's Status column. Three input views: district (12 inputs known without a soil map), field (elevation and soil, 10 inputs) and all (21 inputs).

In [ ]:
import pandas as pd
from IPython.display import Markdown, display

table = pd.read_csv(f"{MODULE}/data/processed/feature_table_v2.csv", low_memory=False)
gps_ok = table["qc_ok"].astype(str).str.strip().str.lower().eq("true")
CROP_LABELS = {"Maize": "Maize", "Beans": "Beans", "Irish potatoes": "Irish potato", "Rice": "Rice"}
train = table[gps_ok & table["Crop_plant"].isin(list(CROP_LABELS))]
print(f"{len(table):,} CIAT fields. {gps_ok.sum():,} pass the GPS checks. "
      f"{len(train):,} of those have one of the four single crops: these are the training fields.")
display(pd.crosstab(train["District"], train["Crop_plant"].map(CROP_LABELS), margins=True, margins_name="Total"))

## 4. Every model of the protocol (the official evaluation)

Five kinds of model go through the same folds: B0 (always the most common crop), B1 (expert rules from published crop ranges, not trained on our data), Logistic Regression, Random Forest and XGBoost, the last three on each input view. Two tests:

- **Main test, a district the model never saw**: train on three districts, test on the fourth, four times. It decides whether a model replaces B1.
- **Second test, new fields in the known districts**: 5 folds, and each 2 km block stays on one side, so neighbouring fields are never split between training and test.

This runs the same command as the official run of 28 Oct, then checks every result file against the committed `reports/eval_v1`. One to three minutes.

In [ ]:
%cd {REPO}
!{PY} 01-crop-recommendation/src/models/evaluate.py --out /content/eval_check > /content/eval_check.log 2>&1 && echo "Evaluation done." || tail -n 30 /content/eval_check.log
!{PY} 01-crop-recommendation/src/models/compare_results.py 01-crop-recommendation/reports/eval_v1 /content/eval_check

### Scores per model

**Main score**: for each district, macro-F1 over the crops with at least 5 fields there, then the mean over the four districts. Macro-F1 gives each crop the same weight, so rice and potato count as much as maize. It goes from 0 to 1. **Adoption rule** (decided before any result): a model replaces B1 only if, on the main test, its mean is higher than B1's and it beats B1 by more than 0.01 in at least 3 of the 4 districts.

In [ ]:
ALGO = {"LR": "Logistic Regression", "RF": "Random Forest", "XGB": "XGBoost"}

def nice(name):
    fixed = {"B0_majority": "B0, most common crop", "B1_expert_rules": "B1, expert rules"}
    if name in fixed:
        return fixed[name]
    algo, _, view = name.partition("[")
    return f"{ALGO.get(algo, algo)}, {view.rstrip(']').replace('_', ' + ')} inputs"

def per_model(path):
    s = pd.read_csv(path)
    wins = [("" if pd.isna(v) else f"{int(v)} of 4") for v in s["districts_beating_B1"]]
    return pd.DataFrame({"Model": s["model"].map(nice), "Main score": s["macro_f1_mean"],
                         "Worst district": s["worst_district_macro_f1"], "Accuracy (mean of 4 districts)": s["top1_mean"],
                         "Top-3 (mean of 4 districts)": s["top3_mean"], "Districts better than B1": wins,
                         "Passes adoption rule": s["passes_rule"].fillna("")})

for title, f in [("Main test: a district the model never saw", "main_lodo_summary.csv"),
                 ("Second test: new fields in the known districts", "main_blockcv_summary.csv")]:
    display(Markdown(f"**{title}**"))
    display(per_model(f"/content/eval_check/{f}").style.format(precision=3).hide(axis="index"))

## 5. Train XGBoost and score it

Same folds and settings as above, for XGBoost on the three input views, with B0 and B1 as reference points. On top of the main score, it gives overall scores, scores per crop and the confusion matrix, checks that every score per district is the same as in `eval_v1`, then trains the final XGBoost on all 1,904 fields. About 15 seconds.

Settings (the protocol's, nothing tuned): 300 trees, depth 4, learning rate 0.05, class-balanced weights, seed 0. Tuning comes after the midterm, inside the training districts only.

In [ ]:
!{PY} 01-crop-recommendation/src/models/train_xgboost.py --out /content/xgboost_v1

### XGBoost scores

How to read them:

- **Main score**: the protocol's score (mean of the four district macro-F1).
- **Accuracy (all fields)**: share of all 1,904 fields where the first crop is the true crop. It mostly reflects Nyagatare, which has 1,084 of the fields. The table of step 4 shows accuracy as the mean of the four districts instead, so the two numbers differ.
- **Balanced accuracy** and **macro-F1 (all fields)**: each crop counts the same.
- **Cohen's kappa**: 0 is chance agreement, 1 is perfect.
- **Top-3**: near 1 for every method, because there are only four crops, so it cannot tell methods apart.

In [ ]:
OUT = "/content/xgboost_v1"
summary = pd.read_csv(f"{OUT}/summary.csv")
overall = pd.read_csv(f"{OUT}/overall.csv")
by_district = pd.read_csv(f"{OUT}/by_district.csv")
by_crop = pd.read_csv(f"{OUT}/by_crop.csv")
check = pd.read_csv(f"{OUT}/check.csv")
TESTS = {"new_district": "Main test: a district the model never saw",
         "new_fields": "Second test: new fields in the known districts"}
CROPS = ["beans", "irish_potato", "maize", "rice"]
CROP_NAMES = {"beans": "Beans", "irish_potato": "Irish potato", "maize": "Maize", "rice": "Rice"}

display(Markdown("#### Overall scores, per model"))
for t, title in TESTS.items():
    o = overall[overall["test"] == t].merge(summary[summary["test"] == t][["model", "districts_beating_B1", "passes_rule"]],
                                           on="model")
    table_t = pd.DataFrame({
        "Model": o["model"].map(nice), "Main score": o["main_score"], "Accuracy (all fields)": o["accuracy"],
        "Balanced accuracy": o["balanced_accuracy"], "Macro-F1 (all fields)": o["macro_f1"],
        "Weighted F1": o["weighted_f1"], "Cohen's kappa": o["cohen_kappa"], "Top-3 (all fields)": o["top3"],
        "Districts better than B1": [("" if pd.isna(v) else f"{int(v)} of 4") for v in o["districts_beating_B1"]],
        "Passes adoption rule": o["passes_rule"].fillna("")}).sort_values("Main score", ascending=False)
    display(Markdown(f"**{title}**"))
    display(table_t.style.format(precision=3).hide(axis="index"))

display(Markdown("#### XGBoost (all inputs) and the expert rules, district by district"))
rows = []
for t, title in TESTS.items():
    d = by_district[by_district["test"] == t].pivot(index="district", columns="model", values="macro_f1")
    for district, r in d.iterrows():
        rows.append({"Test": title, "District": district, "XGBoost (all inputs)": r["XGB[all]"],
                     "Expert rules (B1)": r["B1_expert_rules"], "Difference": r["XGB[all]"] - r["B1_expert_rules"]})
display(pd.DataFrame(rows).style.format(precision=3).hide(axis="index"))

display(Markdown("#### Scores per crop, XGBoost (all inputs)"))
for t, title in TESTS.items():
    c = by_crop[(by_crop["test"] == t) & (by_crop["model"] == "XGB[all]")]
    display(Markdown(f"**{title}**"))
    display(pd.DataFrame({"Crop": c["crop"].map(CROP_NAMES), "Precision": c["precision"], "Recall": c["recall"],
                          "F1": c["f1"], "Fields": c["fields"]}).style.format(precision=3).hide(axis="index"))

same = (check["same_as_eval_v1"] == "yes").all()
print("Every score per district is the same as in the committed eval_v1." if same
      else "Some scores differ from eval_v1: check the library versions printed in step 2.")

### The same scores as charts

Blue is XGBoost on all inputs, orange is the expert rules (B1). Every value is in the tables above.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.patches import Patch
from matplotlib.ticker import PercentFormatter

INK, INK2, GRID, AXIS, SURFACE = "#0b0b0b", "#52514e", "#e1e0d9", "#c3c2b7", "#fcfcfb"
BLUE, ORANGE = "#2a78d6", "#eb6834"
plt.rcParams.update({"font.family": "sans-serif", "font.size": 10, "figure.facecolor": SURFACE,
                     "axes.facecolor": SURFACE, "axes.edgecolor": AXIS, "axes.labelcolor": INK2,
                     "xtick.color": INK2, "ytick.color": INK2, "axes.titlecolor": INK})

def clean(ax, grid_axis="y"):
    for side in ("top", "right", "left"):
        ax.spines[side].set_visible(False)
    ax.tick_params(length=0)
    if grid_axis:
        ax.grid(axis=grid_axis, color=GRID, linewidth=1)
        ax.set_axisbelow(True)

fig, axes = plt.subplots(1, 2, figsize=(12, 4.2), sharey=True)
for ax, (t, title) in zip(axes, TESTS.items()):
    d = by_district[by_district["test"] == t].pivot(index="district", columns="model", values="macro_f1")
    x, w = np.arange(len(d)), 0.3
    ax.bar(x - w / 2 - 0.015, d["XGB[all]"], w, color=BLUE, label="XGBoost (all inputs)")
    ax.bar(x + w / 2 + 0.015, d["B1_expert_rules"], w, color=ORANGE, label="Expert rules (B1)")
    ax.set_xticks(x, d.index)
    ax.set_ylim(0, 1)
    ax.set_yticks(np.arange(0, 1.01, 0.2))
    ax.set_title(title, loc="left", fontsize=11)
    clean(ax)
axes[0].set_ylabel("Main score (macro-F1, 0 to 1)")
fig.legend(*axes[0].get_legend_handles_labels(), loc="upper right", ncol=2, frameon=False)
fig.suptitle("Main score per district", x=0.01, ha="left", fontsize=12, color=INK)
fig.tight_layout(rect=(0, 0, 1, 0.93))
plt.show()

conf = pd.read_csv(f"{OUT}/confusion.csv")
blues = LinearSegmentedColormap.from_list("blues", ["#cde2fb", "#86b6ef", "#3987e5", "#1c5cab", "#104281"])
fig, axes = plt.subplots(1, 2, figsize=(12, 4.8))
for ax, (t, title) in zip(axes, TESTS.items()):
    m = (conf[(conf["test"] == t) & (conf["model"] == "XGB[all]")]
         .pivot(index="true", columns="predicted", values="count").reindex(index=CROPS, columns=CROPS).fillna(0))
    share = m.div(m.sum(axis=1), axis=0)
    ax.imshow(share.values, cmap=blues, vmin=0, vmax=1)
    for i in range(len(CROPS)):
        for j in range(len(CROPS)):
            v = share.values[i, j]
            ax.text(j, i, f"{int(m.values[i, j])}\n{v:.0%}", ha="center", va="center", fontsize=9,
                    color="#ffffff" if v >= 0.6 else INK)
    names = [CROP_NAMES[c] for c in CROPS]
    ax.set_xticks(range(len(CROPS)), names)
    ax.set_yticks(range(len(CROPS)), names)
    ax.set_xlabel("Predicted crop")
    ax.set_ylabel("True crop")
    ax.set_title(title, loc="left", fontsize=11)
    for s in ax.spines.values():
        s.set_visible(False)
    ax.tick_params(length=0)
fig.suptitle("XGBoost (all inputs): fields per true and predicted crop, and the share of each true crop",
             x=0.01, ha="left", fontsize=12, color=INK)
fig.tight_layout(rect=(0, 0, 1, 0.93))
plt.show()

## 6. The final model and what it relies on

The final XGBoost uses all 21 inputs and all 1,904 training fields, with the same settings. It is the model behind the dashboard map (the dashboard trains the same model from the same data and settings when it starts).

Importance here is XGBoost's gain: how much each input improved the trees. Orange inputs have one value for the whole district, blue inputs change from field to field.

In [ ]:
import json
meta = json.load(open(f"{OUT}/model/meta.json"))
print(f"Final model: XGBoost on {len(meta['features'])} inputs, trained on {meta['trained_on_fields']:,} fields, "
      f"season {meta['season']}. Saved in {OUT}/model.")

FEATURE_NAMES = {
    "precip_mm": "Season rainfall (district)", "tmax_c": "Max temperature (district)",
    "tmin_c": "Min temperature (district)", "tmean_c": "Mean temperature (district)",
    "solar_mj_m2": "Sunlight (district)", "tmean_point_c": "Mean temperature at the field's height",
    "tmin_point_c": "Min temperature at the field's height", "elevation_m": "Elevation",
    "nisr_share_beans": "NISR share of beans", "nisr_share_irish_potato": "NISR share of Irish potato",
    "nisr_share_maize": "NISR share of maize", "nisr_share_paddy_rice": "NISR share of rice",
    "nitrogen_0_20cm": "Soil nitrogen", "phosphorous_extractable_0_20cm": "Soil phosphorus",
    "potassium_extractable_0_20cm": "Soil potassium", "ph_0_20cm": "Soil pH", "sand_0_20cm": "Soil sand",
    "clay_0_20cm": "Soil clay", "silt_0_20cm": "Soil silt", "carbon_organic_0_20cm": "Soil organic carbon",
    "cation_exchange_capacity_0_20cm": "Soil CEC"}
imp = pd.read_csv(f"{OUT}/feature_importance.csv")
level = imp.groupby("level")["share"].sum()
print(f"Same for the whole district: {level.get('district', 0):.0%} of the gain. "
      f"Changes from field to field: {level.get('field', 0):.0%}.")

top = imp.head(12).iloc[::-1]
fig, ax = plt.subplots(figsize=(10, 5))
ax.barh(range(len(top)), top["share"], height=0.6,
        color=[ORANGE if lv == "district" else BLUE for lv in top["level"]])
ax.set_yticks(range(len(top)), [FEATURE_NAMES.get(f, f) for f in top["feature"]])
for i, v in enumerate(top["share"]):
    ax.text(v + 0.006, i, f"{v:.0%}", va="center", fontsize=9, color=INK2)
ax.set_xlim(0, top["share"].max() * 1.15)
ax.xaxis.set_major_formatter(PercentFormatter(1.0, decimals=0))
ax.set_xlabel("Share of the model's gain")
ax.legend(handles=[Patch(color=ORANGE, label="Same for the whole district (weather, NISR shares)"),
                   Patch(color=BLUE, label="Changes from field to field (soil, elevation, temperature at the field)")],
          loc="lower right", frameon=False)
ax.set_title("What the final XGBoost relies on (top 12 inputs)", loc="left", fontsize=12)
clean(ax, grid_axis="x")
fig.tight_layout()
plt.show()

## 7. Try points the way the dashboard does

For a clicked point: find the district, take soil and elevation from the CIAT fields in the same 2 km block (or the nearest block in the district), add the district's season weather and NISR shares, then rank the crops with XGBoost and with the rules, each with its tested score in that district.

The points below are centres of 2 km blocks with many CIAT fields, one per district, plus a point in Kigali (not a pilot district). They use season B, the only season in the data. The last call uses today's season, as the dashboard does by default.

In [ ]:
!{PY} 01-crop-recommendation/src/recommendation/map_model.py --season B --point -1.458 29.556 --point -1.584 29.430 --point -1.350 30.258 --point -2.232 29.646 --point -1.950 30.060
print()
!{PY} 01-crop-recommendation/src/recommendation/map_model.py --point -1.350 30.258

## 8. Save the results

Zips the XGBoost scores of this run as `/content/xgboost_v1_colab.zip` (without the model file). To keep them in the repo: download the zip (folder icon on the left, or set `DOWNLOAD = True`), unzip it into `01-crop-recommendation/reports/xgboost_v1/` on your laptop, and commit. The model folder is never committed: it takes seconds to retrain.

In [ ]:
DOWNLOAD = False   # set to True to download the zip to your computer
!cd /content && rm -f xgboost_v1_colab.zip && zip -qr xgboost_v1_colab.zip xgboost_v1 -x "xgboost_v1/model/*" && echo "Saved /content/xgboost_v1_colab.zip"
if DOWNLOAD:
    from google.colab import files
    files.download("/content/xgboost_v1_colab.zip")

## 9. Run all the tests

About a minute. Every test should pass.

In [ ]:
%cd {REPO}
!{PY} -m pytest -q -p no:cacheprovider 2>&1 | tail -n 3

## 10. Open the dashboard

The cell below starts the dashboard on this Colab machine and prints two links:

- **Public link** (Cloudflare quick tunnel): anyone who has it can open the dashboard while this runtime is running. Good for the meeting. Share it only with the team.
- **Private link** (Colab): only for you, while this notebook is open.

Open a link and wait a few seconds the first time (the map page trains XGBoost once). Then click a field inside one of the four green districts. The dashboard keeps running after the cell finishes, until the runtime stops or you use the stop cell below it.

In [ ]:
import re, time, urllib.request

for proc in (globals().get("tunnel"), globals().get("dashboard")):
    if proc is not None and proc.poll() is None:
        proc.terminate()

dashboard = subprocess.Popen(
    [PY, "-m", "streamlit", "run", "app.py", "--server.port", "8501", "--server.headless", "true",
     "--browser.gatherUsageStats", "false"],
    cwd=f"{REPO}/03-shared/dashboard", stdout=open("/content/streamlit.log", "w"), stderr=subprocess.STDOUT)
up = False
for _ in range(90):
    try:
        up = urllib.request.urlopen("http://localhost:8501/_stcore/health", timeout=2).read().strip() == b"ok"
    except Exception:
        up = False
    if up or dashboard.poll() is not None:
        break
    time.sleep(1)
if not up:
    print(open("/content/streamlit.log").read()[-3000:])
    raise RuntimeError("The dashboard did not start. Its log is above.")
print("The dashboard is running on this Colab machine.")

public = None
cf = "/content/cloudflared"
def downloaded():
    return os.path.exists(cf) and os.path.getsize(cf) > 1_000_000
if not downloaded():
    subprocess.run(["wget", "-q", "-O", cf,
                    "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64"])
if downloaded():
    os.chmod(cf, 0o755)
    tunnel = subprocess.Popen([cf, "tunnel", "--url", "http://localhost:8501", "--no-autoupdate"],
                              stdout=open("/content/cloudflared.log", "w"), stderr=subprocess.STDOUT)
    for _ in range(60):
        time.sleep(1)
        found = re.search(r"https://(?!api\.)[a-z0-9-]+\.trycloudflare\.com", open("/content/cloudflared.log").read())
        if found:
            public = found.group(0)
            break
        if tunnel.poll() is not None:   # the tunnel stopped, for example without internet access
            break
print("Public link:", public or "not available right now (use the private link)")

try:
    from google.colab import output
    print("Private link:")
    output.serve_kernel_port_as_window(8501, anchor_text="Open the dashboard (only you)")
except Exception:
    print("Private link: not on Colab, open http://localhost:8501 on this machine.")

In [ ]:
# Stop the dashboard and its links: set STOP_DASHBOARD to True, then run this cell. Run all leaves it running.
STOP_DASHBOARD = False
if STOP_DASHBOARD:
    for proc in (globals().get("tunnel"), globals().get("dashboard")):
        if proc is not None and proc.poll() is None:
            proc.terminate()
    print("Dashboard stopped.")
else:
    print("The dashboard keeps running. To stop it, set STOP_DASHBOARD = True and run this cell, or close the runtime.")